In [8]:
# 1. Setup: GPU check, Drive, data, tokenizer
import torch
from importlib.metadata import version, PackageNotFoundError

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU attached. Use Runtime > Change runtime type > T4 GPU, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0))

for lib in ["torchao", "bitsandbytes", "peft", "transformers"]:
    try:
        print(lib, version(lib))
    except PackageNotFoundError:
        print(lib, "NOT installed")

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from transformers import AutoTokenizer

DATA_DIR = "/content/drive/MyDrive/promt shield"
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAXLEN = 256

df = pd.read_csv(f"{DATA_DIR}/dataset_v2.csv", dtype=str, keep_default_na=False)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
assert set(df["split"]) == {"train", "val"}
df["y"] = (df["final_label"] == "attack").astype(int)
val_df = df[df["split"] == "val"].reset_index(drop=True)
print("val rows:", len(val_df))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print("pad token id:", tok.pad_token_id, "| padding side:", tok.padding_side)

torch: 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4
torchao NOT installed
bitsandbytes 0.50.2
peft 0.21.2
transformers 5.22.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
val rows: 3298
pad token id: 151643 | padding side: right


In [9]:
# 2. Imports and the benchmark prompts (validation rows only)
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

device = torch.device("cuda")
RESULTS_DIR = f"{DATA_DIR}/results"
CKPT_DIR = f"{DATA_DIR}/checkpoints"

bench_df = val_df.sample(640, random_state=0).reset_index(drop=True)
bench_texts = bench_df["text"].tolist()
print("benchmark prompts:", len(bench_texts), "| GPU:", torch.cuda.get_device_name(0))
print("GPU memory in use at the start (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

benchmark prompts: 640 | GPU: Tesla T4
GPU memory in use at the start (GB): 3.1


In [10]:
# 3. Measuring functions
def make_batch(id_lists, pad_id):
    seqs = [torch.tensor(x) for x in id_lists]
    lengths = torch.tensor([len(s) for s in seqs])
    input_ids = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    attention_mask = (torch.arange(input_ids.shape[1])[None, :] < lengths[:, None]).long()
    return input_ids.to(device), attention_mask.to(device)


def forward_probs(model, id_lists, pad_id, batch_size):
    probs = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(id_lists), batch_size):
            ids, mask = make_batch(id_lists[i:i + batch_size], pad_id)
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    return np.array(probs)


def tokenize_all(tok_, texts):
    return [tok_(t[:4000], truncation=True, max_length=MAXLEN)["input_ids"] for t in texts]


def check_against_saved(model, tok_, saved_csv, n=200):
    saved = pd.read_csv(saved_csv).set_index("id")["prob_attack"]
    sub = bench_df.iloc[:n]
    now = forward_probs(model, tokenize_all(tok_, sub["text"]), tok_.pad_token_id, 32)
    diff = np.abs(now - saved.loc[sub["id"]].values)
    print(f"   check against the probabilities saved during training ({n} prompts): "
          f"mean abs difference {diff.mean():.4f} | max {diff.max():.4f}")


def dir_size_mb(path):
    total = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path))
    return round(total / 1e6, 1)


def bench(tag, model, tok_, mem_before):
    model.eval()
    pad_id = tok_.pad_token_id
    id_lists = tokenize_all(tok_, bench_texts)
    loaded_gb = torch.cuda.memory_allocated() / 1e9 - mem_before

    for ids in id_lists[:10]:
        forward_probs(model, [ids], pad_id, 1)
    times = []
    for ids in id_lists[:100]:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        forward_probs(model, [ids], pad_id, 1)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    forward_probs(model, id_lists[:64], pad_id, 32)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    forward_probs(model, id_lists, pad_id, 32)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 - mem_before

    return {
        "model": tag,
        "loaded_gb": round(loaded_gb, 2),
        "peak_gb_batch32": round(peak_gb, 2),
        "latency_median_ms": round(float(np.median(times)), 1),
        "latency_p95_ms": round(float(np.percentile(times, 95)), 1),
        "prompts_per_s_batch32": round(len(id_lists) / seconds, 1),
        "median_tokens": int(np.median([len(x) for x in id_lists])),
    }

In [11]:
# 4. Model loaders
def load_distilbert():
    tok_ = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    m = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2, dtype=torch.float16
    ).to(device)
    return m, tok_


def load_lora(path):
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, dtype=torch.float16)
    base.config.pad_token_id = tok.pad_token_id
    m = PeftModel.from_pretrained(base.to(device), path)
    return m, tok


def load_qlora(path):
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, quantization_config=bnb_cfg, dtype=torch.float16, device_map={"": 0},
    )
    base.config.pad_token_id = tok.pad_token_id
    m = PeftModel.from_pretrained(base, path)
    return m, tok

In [12]:
# 5. Run one model: load, check, measure, free the GPU
results = []


def run(tag, build, saved_csv=None, adapter_dir=None):
    gc.collect()
    torch.cuda.empty_cache()
    before = torch.cuda.memory_allocated() / 1e9
    t0 = time.time()
    model, tok_ = build()
    torch.cuda.synchronize()
    load_s = time.time() - t0
    print(tag, "| loaded in", round(load_s, 1), "s")
    if saved_csv:
        check_against_saved(model, tok_, saved_csv)
    row = bench(tag, model, tok_, before)
    row["load_seconds"] = round(load_s, 1)
    row["adapter_mb_on_disk"] = dir_size_mb(adapter_dir) if adapter_dir else None
    results.append(row)
    print("  ", row)
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [13]:
# 6. Run the benchmark
run("distilbert", load_distilbert)
run("lora", lambda: load_lora(f"{CKPT_DIR}/lora_epoch2"),
    saved_csv=f"{RESULTS_DIR}/lora_val_probs_epoch2.csv", adapter_dir=f"{CKPT_DIR}/lora_epoch2")
run("qlora", lambda: load_qlora(f"{CKPT_DIR}/qlora_epoch1"),
    saved_csv=f"{RESULTS_DIR}/qlora_val_probs_epoch1.csv", adapter_dir=f"{CKPT_DIR}/qlora_epoch1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distilbert | loaded in 2.2 s
   {'model': 'distilbert', 'loaded_gb': 0.15, 'peak_gb_batch32': 0.33, 'latency_median_ms': 7.5, 'latency_p95_ms': 14.7, 'prompts_per_s_batch32': 660.1, 'median_tokens': 81, 'load_seconds': 2.2, 'adapter_mb_on_disk': None}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lora | loaded in 14.3 s
   check against the probabilities saved during training (200 prompts): mean abs difference 0.0000 | max 0.0000
   {'model': 'lora', 'loaded_gb': 3.11, 'peak_gb_batch32': 3.88, 'latency_median_ms': 60.3, 'latency_p95_ms': 74.1, 'prompts_per_s_batch32': 25.0, 'median_tokens': 78, 'load_seconds': 14.3, 'adapter_mb_on_disk': 17.5}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


qlora | loaded in 14.8 s
   check against the probabilities saved during training (200 prompts): mean abs difference 0.0000 | max 0.0000
   {'model': 'qlora', 'loaded_gb': 1.2, 'peak_gb_batch32': 1.98, 'latency_median_ms': 97.3, 'latency_p95_ms': 129.6, 'prompts_per_s_batch32': 21.9, 'median_tokens': 78, 'load_seconds': 14.8, 'adapter_mb_on_disk': 17.5}


In [16]:
# 7. Results table, saved to Drive
bench_table = pd.DataFrame(results)
bench_table.to_csv(f"{RESULTS_DIR}/inference_benchmark.csv", index=False)
print("GPU:", torch.cuda.get_device_name(0))
print(bench_table.to_string(index=False))

GPU: Tesla T4
     model  loaded_gb  peak_gb_batch32  latency_median_ms  latency_p95_ms  prompts_per_s_batch32  median_tokens  load_seconds  adapter_mb_on_disk
distilbert       0.15             0.33                7.5            14.7                  660.1             81           2.2                 NaN
      lora       3.11             3.88               60.3            74.1                   25.0             78          14.3                17.5
     qlora       1.20             1.98               97.3           129.6                   21.9             78          14.8                17.5
